<a href="https://colab.research.google.com/github/santidrelew/tp-mpg/blob/main/notebooks/02_extraccion_muestra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **2 - Extracción de la muestra**

In [2]:
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    !git clone https://github.com/santidrelew/tp-mpg.git
    %cd tp-mpg

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

In [ ]:
# Cargamos los datasets
PROJECT_DIR = Path().resolve() if EN_COLAB else Path().resolve().parent
DATA_BASE_DIR = PROJECT_DIR / "data" / "procesado"

df = pd.read_csv(DATA_BASE_DIR / "milan_airbnb_filtrado.csv")

Vamos a hacer un muestreo estratificado tomando como estrato la variable neighbourhood_cleansed. Esto es porque las tarifas y amenidades difieren sustancialmente entre el centro historico y las comunas perifericas. Un muestreo puramente aleatorio corria el riesgo de subestimar zonas de menor densidad de oferta (como la comuna 7).

In [5]:
N_MUESTRA = 750
fraccion = N_MUESTRA / len(df)

df_muestra = df.groupby("neighbourhood_cleansed", group_keys=False).apply(
    lambda x: x.sample(frac=fraccion, random_state=42)
)

# Redondeamos
if len(df_muestra) > N_MUESTRA:
  df_muestra = df_muestra.sample(n=N_MUESTRA, random_state=42)

In [11]:
df_muestra

,host_is_superhost,host_total_listings_count,host_identity_verified,latitude,longitude,accommodates,bathrooms,bedrooms,guests_included,extra_people,...,Iron,Host_greets_you,Paid_parking_on_premises,Luggage_dropoff_allowed,Long_term_stays_allowed,Doorman,Pets_allowed,Smoking_allowed,check_in_24_hour,log_daily_price
677,0,2,0,45.46886,9.20545,3,4,1,1,0,...,0,0,0,0,0,0,1,0,0,5.017280
1591,0,4,0,45.46461,9.18406,2,3,1,1,0,...,1,0,0,0,0,0,1,0,0,4.795791
984,0,41,0,45.46390,9.19402,6,3,1,3,9,...,1,1,0,0,0,0,0,0,0,4.394449
607,0,410,0,45.46366,9.19744,4,3,1,2,26,...,1,0,0,0,0,0,0,0,0,4.912655
1392,0,8,0,45.46186,9.19247,6,3,2,1,0,...,0,0,0,1,0,0,1,0,0,5.398163
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9097,1,1,1,45.48082,9.19090,6,5,3,4,30,...,1,1,0,0,0,0,0,0,0,5.017280
8530,1,6,1,45.48204,9.18836,4,3,0,2,15,...,0,0,0,0,0,1,0,0,0,4.382027
8540,0,1,1,45.51777,9.19138,4,3,1,2,10,...,0,0,0,0,0,0,0,0,0,4.110874
9196,0,1,0,45.50033,9.17697,3,3,2,2,30,...,0,0,0,0,0,0,0,0,0,3.828641


In [7]:
TARGET = "log_daily_price"

media_pop = df[TARGET].mean()
var_pop = df[TARGET].var()

media_mue = df_muestra[TARGET].mean()
var_mue = df_muestra[TARGET].var()

tabla_validacion = pd.DataFrame(
    {
        "Metrica": [
            "Media (€)",
            "Varianza",
            "Desvio Estándar (€)",
            "Mediana (€)",
        ],
        "Poblacion": [
            round(media_pop, 2),
            round(var_pop, 2),
            round(df[TARGET].std(), 2),
            round(df[TARGET].median(), 2),
        ],
        "Muestra (n=750)": [
            round(media_mue, 2),
            round(var_mue, 2),
            round(df_muestra[TARGET].std(), 2),
            round(df_muestra[TARGET].median(), 2),
        ],
        "Diferencia Relativa (%)": [
            round(abs(media_pop - media_mue) / media_pop * 100, 2),
            round(abs(var_pop - var_mue) / var_pop * 100, 2),
            round(
                abs(df[TARGET].std() - df_muestra[TARGET].std())
                / df[TARGET].std()
                * 100,
                2,
            ),
            round(
                abs(df[TARGET].median() - df_muestra[TARGET].median())
                / df[TARGET].median()
                * 100,
                2,
            ),
        ],
    }
)

display(tabla_validacion)

,Metrica,Poblacion,Muestra (n=750),Diferencia Relativa (%)
0,Media (€),4.44,4.42,0.46
1,Varianza,0.30,0.27,10.30
2,Desvio Estándar (€),0.55,0.52,5.29
3,Mediana (€),4.33,4.33,0.00


# ANALZAR OTRA VEZ

Como tanto la media como la mediana tienen una diferencia relativa de menos del 5% (la mediana, que es una medida robusta frente a la asimetria, es exactamente la misma), podemos decir que la muestra es representativa.

En cuanto a la varianza y el desvio estandar, la gran diferencia relativa no representa un sesgo de muestreo, sino que la poblacion total contiene valores extremos y atipicos en la cola superior (como las propiedades de $3.000$ €) que inflan la dispersion global. Al extraer una muestra aleatoria estratificada, la probabilidad de capturar dichos eventos aislados disminuye, lo que reduce la varianza muestral. A su vez, justifican el uso de log_daily_price para neutralizar los valores extremos en nuestra regresion.

In [10]:
df_muestra["neighbourhood_cleansed"]

KeyError: 'neighbourhood_cleansed'

In [8]:
zona = "neighbourhood_cleansed"
proporciones = pd.DataFrame({
    "Base (%)": df[zona].value_counts(normalize=True) * 100,
    "Muestra (%)": df_muestra[zona].value_counts(normalize=True) * 100,
}).fillna(0).sort_index()
proporciones["Diferencia (p. p.)"] = (
    proporciones["Muestra (%)"] - proporciones["Base (%)"]
)
display(proporciones.round(2))

KeyError: 'neighbourhood_cleansed'

In [ ]:
# Guardamos la muestra
df_muestra.to_csv(DATA_BASE_DIR / "milan_airbnb_muestra.csv", index=False)